# ERP003950 (мышь) — PCR-like amplification → fragmentation → InSilicoSeq 2x150

Обновлённая копия `simulate_mouse_merged_insilicoseq_150bp.ipynb`: сначала моделирует
sample/template-level amplification bias, затем случайную нарезку full-length 5′RACE-derived
V(D)J templates и paired-end sequencing 2x150 через InSilicoSeq.

Реалистичный wet-lab порядок для такой постановки: RT/template switching → PCR amplification
full-length 5′RACE molecules → fragmentation/tagmentation/library-prep PCR → paired-end sequencing.
Здесь явно моделируется главный эффект amplification до fragmentation: неравномерное abundance
между templates (PCR jackpot/overdispersion), а сами reads затем раскладываются по template
через `iss generate --sequence_type metagenomics`.

Что переиспользуется, если уже есть:

- `SHARED_TEMPLATES_DIR/*_templates.fasta` из старого fixed-end ISS прогона — сами merged templates.
- `SHARED_TEMPLATES_DIR/*_read_counts.tsv` — исходное template abundance до новой PCR-like модели.
- 150bp custom error model из `insilicoseq_150bp/model/*.npz`, если он уже создан.

Что создаётся отдельно и не перезаписывает старые неподходящие результаты:

- `/data/user/epishkin/results/ERP003950/simulated/insilicoseq_150bp_pcr_fragmented/`
- `read_counts/*_read_counts_pcr_fragmented.tsv`
- `fastq/{sample}_R1/_R2.fastq.gz`
- `qc/amplification_qc.tsv`, `qc/simulate_qc.tsv`

Kernel: **BCR Pipeline** (`bcr_env`), OneQ task `bbc68913-4463-433d-b647-c3b8a76555ba`.


### 1. Env check

In [1]:
import os, sys, sysconfig, subprocess

_ENV_CANDIDATES = [
    "/data/user/epishkin/conda/envs/bcr_env",
    "/opt/conda/envs/bcr_env",
]
_CONDA_ENV = next((p for p in _ENV_CANDIDATES if os.path.isdir(p + "/bin")), _ENV_CANDIDATES[-1])
os.environ["PATH"] = _CONDA_ENV + "/bin:" + os.environ.get("PATH", "")
os.environ["PYTHONNOUSERSITE"] = "1"
sys.path[:] = [p for p in sys.path if "/data/user/epishkin/.local" not in p]
for _site in [
    _CONDA_ENV + "/lib/python3.11/site-packages",
    _CONDA_ENV + "/lib/python3.12/site-packages",
    sysconfig.get_path("purelib"),
]:
    if os.path.isdir(_site) and _site not in sys.path:
        sys.path.insert(0, _site)
os.environ["HOME"] = "/data/user/epishkin"
os.environ["XDG_CONFIG_HOME"] = "/data/user/epishkin/.config"
os.makedirs(os.environ["XDG_CONFIG_HOME"], exist_ok=True)

print(f"Using env: {_CONDA_ENV}")
for tool in ("iss", "bowtie2", "bowtie2-build", "samtools"):
    path = subprocess.run(["which", tool], capture_output=True, text=True).stdout.strip()
    if not path:
        raise RuntimeError(
            f"'{tool}' not found in PATH. Install with:\n"
            f"  conda install -c bioconda -c conda-forge bowtie2 samtools -y  # (iss should already be present)"
        )
    print(tool, "->", path)
print(subprocess.run(["iss", "--version"], capture_output=True, text=True).stderr.strip())


Using env: /data/user/epishkin/conda/envs/bcr_env
iss -> /data/user/epishkin/conda/envs/bcr_env/bin/iss
bowtie2 -> /data/user/epishkin/conda/envs/bcr_env/bin/bowtie2
bowtie2-build -> /data/user/epishkin/conda/envs/bcr_env/bin/bowtie2-build
samtools -> /data/user/epishkin/conda/envs/bcr_env/bin/samtools



### 2. Config

In [2]:
from pathlib import Path

VOLUME = Path("/data/user/epishkin")
DATASET = "ERP003950"
SAMPLES = ["ERR346596", "ERR346597", "ERR346598", "ERR346599", "ERR346600", "ERR346601"]

MERGED_FASTQ_DIR = VOLUME / "results" / DATASET / "merged" / "fastq"
RAW_FASTQ_DIR = VOLUME / "raw" / DATASET  # ERR346596_1.fastq.gz / _2.fastq.gz

# Templates + original abundance from the old fixed-end amplicon ISS prep.
# 150bp notebook uses *_templates.fasta from here; this updated notebook also uses
# *_read_counts.tsv as the starting molecular/template abundance before PCR-like overdispersion.
SHARED_TEMPLATES_DIR = VOLUME / "results" / DATASET / "simulated" / "insilicoseq" / "templates"

TARGET_READ_LENGTH = 150

# New, isolated output tree. Do not overwrite insilicoseq_150bp or old fixed-end outputs.
OUT_BASE = VOLUME / "results" / DATASET / "simulated" / f"insilicoseq_{TARGET_READ_LENGTH}bp_pcr_fragmented"
MODEL_DIR = OUT_BASE / "model"          # used only if existing 150bp model is absent
COUNTS_DIR = OUT_BASE / "read_counts"   # PCR-like counts for fragmented simulation
FASTQ_DIR = OUT_BASE / "fastq"
LOGS_DIR = OUT_BASE / "logs"
QC_DIR = OUT_BASE / "qc"
for d in (MODEL_DIR, COUNTS_DIR, FASTQ_DIR, LOGS_DIR, QC_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Prefer reusing the 150bp error model already trained by simulate_mouse_merged_insilicoseq_150bp.ipynb.
EXISTING_150BP_MODEL_DIR = VOLUME / "results" / DATASET / "simulated" / f"insilicoseq_{TARGET_READ_LENGTH}bp" / "model"
EXISTING_CUSTOM_MODEL_PREFIX = EXISTING_150BP_MODEL_DIR / f"ERP003950_mouse_miseq{TARGET_READ_LENGTH}"
EXISTING_CUSTOM_MODEL_NPZ = Path(str(EXISTING_CUSTOM_MODEL_PREFIX) + ".npz")

REF_FASTA = MODEL_DIR / "ERP003950_all_samples_reference.fasta"
BOWTIE2_INDEX = MODEL_DIR / "ERP003950_bt2_index"
BAM_PATH = MODEL_DIR / "ERP003950_real_reads_vs_merged.bam"
OWN_CUSTOM_MODEL_PREFIX = MODEL_DIR / f"ERP003950_mouse_miseq{TARGET_READ_LENGTH}"
OWN_CUSTOM_MODEL_NPZ = Path(str(OWN_CUSTOM_MODEL_PREFIX) + ".npz")

REUSE_EXISTING_150BP_MODEL = EXISTING_CUSTOM_MODEL_NPZ.exists()
CUSTOM_MODEL_PREFIX = EXISTING_CUSTOM_MODEL_PREFIX if REUSE_EXISTING_150BP_MODEL else OWN_CUSTOM_MODEL_PREFIX
CUSTOM_MODEL_NPZ = EXISTING_CUSTOM_MODEL_NPZ if REUSE_EXISTING_150BP_MODEL else OWN_CUSTOM_MODEL_NPZ

print("OUT_BASE:", OUT_BASE)
print("SHARED_TEMPLATES_DIR:", SHARED_TEMPLATES_DIR)
print("REUSE_EXISTING_150BP_MODEL:", REUSE_EXISTING_150BP_MODEL)
print("CUSTOM_MODEL_NPZ:", CUSTOM_MODEL_NPZ)

# --- runtime knobs ---
NPROC = 8
SEED = 42
COMPRESS = True
FORCE = False

# --- PCR-like amplification + fragmentation knobs ---
SEQUENCE_TYPE = "metagenomics"
TARGET_COVERAGE = 8.0
FRAGMENT_LENGTH_MEAN = 200
FRAGMENT_LENGTH_SD = 40

# PCR-like overdispersion: multiplier ~ lognormal(-sigma^2/2, sigma), so mean multiplier ~1.
# Increase PCR_LOGNORMAL_SIGMA for stronger jackpotting between templates.
PCR_LOGNORMAL_SIGMA = 1.0
MIN_READ_PAIRS_PER_TEMPLATE = 1  # keep every template represented; set 0 for harsher sampling/dropout


OUT_BASE: /data/user/epishkin/results/ERP003950/simulated/insilicoseq_150bp_pcr_fragmented
SHARED_TEMPLATES_DIR: /data/user/epishkin/results/ERP003950/simulated/insilicoseq/templates
REUSE_EXISTING_150BP_MODEL: True
CUSTOM_MODEL_NPZ: /data/user/epishkin/results/ERP003950/simulated/insilicoseq_150bp/model/ERP003950_mouse_miseq150.npz


### 3. Train the 150bp error model

`iss model` сам не выравнивает — нужен готовый BAM реальных reads на
референс. Референс — уже собранные (`AssemblePairs.py`) merged sequences
этого же датасета: выравниваем raw R1/R2 обратно на них через `bowtie2`.
Раз реальный ERP003950 — честный 2x250, а нужна модель на 150bp — raw reads
перед выравниванием обрезаются `bowtie2 --trim-to TARGET_READ_LENGTH` (с 3'-конца,
т.е. остаётся 5'-часть рида, где качество выше). Референс и bowtie2-индекс
(шаги 3a/3b) от целевой длины рида не зависят и не пересобираются.

Тяжёлый шаг (~4.6M read pairs по всем 6 samples) — heartbeat каждые 30с.


In [3]:
import time

def run_with_heartbeat(cmd, log_path, heartbeat=30, shell=False):
    t0 = time.time()
    with open(log_path, "w") as log_h:
        proc = subprocess.Popen(cmd, stdout=log_h, stderr=subprocess.STDOUT, text=True, shell=shell)
        label = cmd if shell else " ".join(cmd)
        print(f"[run] {label}\n  pid={proc.pid} log={log_path}")
        while proc.poll() is None:
            print(f"  still running: pid={proc.pid} elapsed={(time.time()-t0)/60:.1f} min", flush=True)
            time.sleep(heartbeat)
    elapsed = time.time() - t0
    if proc.returncode != 0:
        raise RuntimeError(f"command failed (exit {proc.returncode}); see {log_path}")
    print(f"done: elapsed={elapsed/60:.1f} min")

def iter_fastq(path):
    import gzip
    opener = gzip.open if str(path).endswith(".gz") else open
    with opener(path, "rt") as h:
        while True:
            head = h.readline()
            if not head:
                return
            seq = h.readline().rstrip("\n")
            h.readline()  # +
            h.readline()  # качество
            yield seq

def iter_fasta(path):
    with open(path) as h:
        header, seq_chunks = None, []
        for line in h:
            line = line.rstrip("\n")
            if line.startswith(">"):
                if header is not None:
                    yield header, "".join(seq_chunks)
                header, seq_chunks = line[1:], []
            else:
                seq_chunks.append(line)
        if header is not None:
            yield header, "".join(seq_chunks)


In [4]:
if REUSE_EXISTING_150BP_MODEL:
    print(f'[skip] reusing existing 150bp ISS model: {CUSTOM_MODEL_NPZ}')
else:
    # 3a. Собрать один reference FASTA из merged (assemble-pass) reads всех samples
    def merged_fastq_to_fasta(sample, out_handle):
        fq = MERGED_FASTQ_DIR / f"{sample}_assemble-pass.fastq.gz"
        n = 0
        for i, seq in enumerate(iter_fastq(fq)):
            out_handle.write(f">{sample}_{i}\n{seq}\n")
            n += 1
        return n

    if REF_FASTA.exists() and not FORCE:
        print(f"[skip] {REF_FASTA} exists")
    else:
        total = 0
        with open(REF_FASTA, "w") as out_h:
            for sample in SAMPLES:
                n = merged_fastq_to_fasta(sample, out_h)
                total += n
                print(f"  {sample}: {n:,} reference sequences")
        print(f"wrote {REF_FASTA} ({total:,} sequences total)")


[skip] reusing existing 150bp ISS model: /data/user/epishkin/results/ERP003950/simulated/insilicoseq_150bp/model/ERP003950_mouse_miseq150.npz


In [5]:
if REUSE_EXISTING_150BP_MODEL:
    print(f'[skip] reusing existing 150bp ISS model: {CUSTOM_MODEL_NPZ}')
else:
    # 3b. bowtie2-build
    index_done_marker = Path(str(BOWTIE2_INDEX) + ".1.bt2")
    if index_done_marker.exists() and not FORCE:
        print(f"[skip] bowtie2 index already built: {BOWTIE2_INDEX}")
    else:
        run_with_heartbeat(
            ["bowtie2-build", "--threads", str(NPROC), str(REF_FASTA), str(BOWTIE2_INDEX)],
            LOGS_DIR / "bowtie2_build.log",
        )


[skip] reusing existing 150bp ISS model: /data/user/epishkin/results/ERP003950/simulated/insilicoseq_150bp/model/ERP003950_mouse_miseq150.npz


In [6]:
if REUSE_EXISTING_150BP_MODEL:
    print(f'[skip] reusing existing 150bp ISS model: {CUSTOM_MODEL_NPZ}')
else:
    # 3c. Выровнять raw R1/R2 (все 6 samples) на merged-read референс -> отсортированный+индексированный BAM
    if BAM_PATH.exists() and not FORCE:
        print(f"[skip] {BAM_PATH} exists")
    else:
        r1_list = ",".join(str(RAW_FASTQ_DIR / f"{s}_1.fastq.gz") for s in SAMPLES)
        r2_list = ",".join(str(RAW_FASTQ_DIR / f"{s}_2.fastq.gz") for s in SAMPLES)
        align_cmd = (
            f"bowtie2 --local --trim-to {TARGET_READ_LENGTH} -p {NPROC} -x {BOWTIE2_INDEX} -1 {r1_list} -2 {r2_list} "
            f"2> {LOGS_DIR / 'bowtie2_align.log'} "
            f"| samtools view -bS - "
            f"| samtools sort -@ {NPROC} -o {BAM_PATH} -"
        )
        run_with_heartbeat(align_cmd, LOGS_DIR / "bowtie2_align_pipe.log", shell=True)
        subprocess.run(["samtools", "index", str(BAM_PATH)], check=True)
        print(f"indexed {BAM_PATH}")


[skip] reusing existing 150bp ISS model: /data/user/epishkin/results/ERP003950/simulated/insilicoseq_150bp/model/ERP003950_mouse_miseq150.npz


In [7]:
# 3d. iss model: build custom KDE error model only if the reusable 150bp model is absent.
#
# IMPORTANT: iss/app.py can swallow AttributeError in bam.to_model() and exit 0 without writing .npz.
# Therefore we verify that CUSTOM_MODEL_NPZ exists after the command.
if REUSE_EXISTING_150BP_MODEL:
    print(f"[skip] reuse model: {CUSTOM_MODEL_NPZ}")
elif CUSTOM_MODEL_NPZ.exists() and not FORCE:
    print(f"[skip] {CUSTOM_MODEL_NPZ} exists")
else:
    run_with_heartbeat(
        ["iss", "model", "--debug", "-b", str(BAM_PATH), "-o", str(CUSTOM_MODEL_PREFIX)],
        LOGS_DIR / "iss_model.log",
    )
    if not CUSTOM_MODEL_NPZ.exists():
        log_tail = (LOGS_DIR / "iss_model.log").read_text()[-3000:]
        raise RuntimeError(
            f"iss model exited 0 but did not write {CUSTOM_MODEL_NPZ}. "
            f"Known ISS bug: AttributeError inside bam.to_model() is swallowed. Log tail:\n{log_tail}"
        )

from iss.error_models.kde import KDErrorModel
em = KDErrorModel(str(CUSTOM_MODEL_NPZ))
print(f"model: {CUSTOM_MODEL_NPZ}")
print(f"read_length = {em.read_length}")
if em.read_length != TARGET_READ_LENGTH:
    print(f"WARNING: expected read_length={TARGET_READ_LENGTH}, model actually has {em.read_length}")


[skip] reuse model: /data/user/epishkin/results/ERP003950/simulated/insilicoseq_150bp/model/ERP003950_mouse_miseq150.npz
model: /data/user/epishkin/results/ERP003950/simulated/insilicoseq_150bp/model/ERP003950_mouse_miseq150.npz
read_length = 150


### 4. Построить PCR-like `read_counts.tsv` для fragmentation

Берём два input из `SHARED_TEMPLATES_DIR`:

- `{sample}_templates.fasta` — сами full-length merged templates;
- `{sample}_read_counts.tsv` — исходное abundance из real merged reads / fixed-end amplicon prep.

Дальше моделируем amplification до fragmentation: каждому template задаём PCR-like
lognormal multiplier (`PCR_LOGNORMAL_SIGMA`) и распределяем sample-level target number
of read pairs пропорционально `original_count * multiplier * template_length`.

Итоговые файлы пишутся отдельно:
`insilicoseq_150bp_pcr_fragmented/read_counts/*_read_counts_pcr_fragmented.tsv`.


In [8]:
import csv, math, random

def load_original_counts(path):
    counts = {}
    with open(path) as h:
        for line in h:
            if not line.strip():
                continue
            tpl_id, n = line.rstrip("\n").split("\t")[:2]
            counts[tpl_id] = int(float(n))
    return counts

def build_pcr_fragmented_read_counts(
    sample,
    target_coverage=TARGET_COVERAGE,
    read_length=TARGET_READ_LENGTH,
    pcr_sigma=PCR_LOGNORMAL_SIGMA,
    min_pairs=MIN_READ_PAIRS_PER_TEMPLATE,
    seed=SEED,
    force=FORCE,
):
    templates_fa = SHARED_TEMPLATES_DIR / f"{sample}_templates.fasta"
    original_counts_tsv = SHARED_TEMPLATES_DIR / f"{sample}_read_counts.tsv"
    out_tsv = COUNTS_DIR / f"{sample}_read_counts_pcr_fragmented.tsv"

    if out_tsv.exists() and not force:
        n = sum(1 for _ in open(out_tsv))
        print(f"[{sample}] [skip] {out_tsv.name} exists: {n:,} templates")
        return {"sample": sample, "status": "skipped", "read_counts_tsv": str(out_tsv)}

    if not templates_fa.exists():
        raise FileNotFoundError(f"Missing templates for {sample}: {templates_fa}")
    if not original_counts_tsv.exists():
        raise FileNotFoundError(f"Missing original fixed-end read_counts for {sample}: {original_counts_tsv}")

    original_counts = load_original_counts(original_counts_tsv)
    rng = random.Random((seed or 0) + sum(ord(c) for c in sample))
    mu = -0.5 * (pcr_sigma ** 2)  # lognormal mean ~1

    rows = []
    baseline_total_pairs = 0
    original_total = 0
    for tpl_id, seq in iter_fasta(templates_fa):
        length = len(seq)
        original_n = original_counts.get(tpl_id)
        if original_n is None:
            raise KeyError(f"{sample}: template {tpl_id!r} is absent from {original_counts_tsv}")
        multiplier = rng.lognormvariate(mu, pcr_sigma) if pcr_sigma > 0 else 1.0
        weight = max(0.0, original_n * multiplier * length)
        baseline_pairs = max(1, math.ceil(target_coverage * length / (2 * read_length)))
        baseline_total_pairs += baseline_pairs
        original_total += original_n
        rows.append({
            "tpl_id": tpl_id,
            "length": length,
            "original_n": original_n,
            "multiplier": multiplier,
            "weight": weight,
        })

    total_weight = sum(r["weight"] for r in rows)
    if total_weight <= 0:
        raise RuntimeError(f"{sample}: total PCR-like weight is zero")

    # Deterministic multinomial-like allocation with fractional remainder.
    allocated = 0
    for r in rows:
        exact = baseline_total_pairs * r["weight"] / total_weight
        n_pairs = math.floor(exact)
        if min_pairs and n_pairs < min_pairs:
            n_pairs = min_pairs
        r["exact_pairs"] = exact
        r["n_pairs"] = n_pairs
        allocated += n_pairs

    remaining = baseline_total_pairs - allocated
    if remaining > 0:
        # Add leftover pairs to largest fractional remainders.
        ranked = sorted(rows, key=lambda r: (r["exact_pairs"] - math.floor(r["exact_pairs"])), reverse=True)
        for r in ranked[:remaining]:
            r["n_pairs"] += 1
    elif remaining < 0 and min_pairs == 0:
        # Remove excess from smallest fractional remainders where possible.
        ranked = sorted(rows, key=lambda r: (r["exact_pairs"] - math.floor(r["exact_pairs"])))
        need = -remaining
        for r in ranked:
            if need <= 0:
                break
            drop = min(r["n_pairs"], need)
            r["n_pairs"] -= drop
            need -= drop

    final_total_pairs = sum(r["n_pairs"] for r in rows)
    nonzero_templates = sum(1 for r in rows if r["n_pairs"] > 0)

    with open(out_tsv, "w", newline="") as f:
        writer = csv.writer(f, delimiter="	")
        for r in rows:
            if r["n_pairs"] > 0:
                writer.writerow([r["tpl_id"], r["n_pairs"]])

    print(
        f"[{sample}] templates={len(rows):,} nonzero={nonzero_templates:,} "
        f"original_reads={original_total:,} target_pairs={baseline_total_pairs:,} "
        f"final_pairs={final_total_pairs:,} pcr_sigma={pcr_sigma}"
    )
    return {
        "sample": sample,
        "status": "built",
        "templates": len(rows),
        "nonzero_templates": nonzero_templates,
        "original_reads": original_total,
        "target_read_pairs": baseline_total_pairs,
        "final_read_pairs": final_total_pairs,
        "pcr_lognormal_sigma": pcr_sigma,
        "min_read_pairs_per_template": min_pairs,
        "read_counts_tsv": str(out_tsv),
    }


In [9]:
amplification_rows = [build_pcr_fragmented_read_counts(sample) for sample in SAMPLES]

qc_path = QC_DIR / "amplification_qc.tsv"
built_rows = [r for r in amplification_rows if r.get("status") == "built"]
if built_rows:
    with open(qc_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(built_rows[0].keys()), delimiter="	")
        writer.writeheader()
        writer.writerows(built_rows)
    print(f"wrote {qc_path}")


[ERR346596] templates=624,945 nonzero=624,945 original_reads=658,771 target_pairs=6,426,862 final_pairs=6,426,862 pcr_sigma=1.0
[ERR346597] templates=595,793 nonzero=595,793 original_reads=627,705 target_pairs=6,123,608 final_pairs=6,123,608 pcr_sigma=1.0
[ERR346598] templates=1,302,064 nonzero=1,302,064 original_reads=1,325,428 target_pairs=13,383,820 final_pairs=13,383,820 pcr_sigma=1.0
[ERR346599] templates=713,450 nonzero=713,450 original_reads=722,057 target_pairs=7,337,696 final_pairs=7,337,696 pcr_sigma=1.0
[ERR346600] templates=552,109 nonzero=552,109 original_reads=580,241 target_pairs=5,678,049 final_pairs=5,678,049 pcr_sigma=1.0
[ERR346601] templates=689,729 nonzero=689,729 original_reads=698,208 target_pairs=7,092,671 final_pairs=7,092,671 pcr_sigma=1.0
wrote /data/user/epishkin/results/ERP003950/simulated/insilicoseq_150bp_pcr_fragmented/qc/amplification_qc.tsv


### 5. Run `iss generate` per sample (PCR-like abundance + fragmentation/`metagenomics`, 150bp model)

`--genomes` берёт shared templates, а `--readcount_file` берёт новые PCR-like counts
из `insilicoseq_150bp_pcr_fragmented/read_counts/`. Выход пишется только в новую
папку `insilicoseq_150bp_pcr_fragmented/fastq/`.


In [10]:
for sample in SAMPLES:
    fa = SHARED_TEMPLATES_DIR / f"{sample}_templates.fasta"
    tsv = COUNTS_DIR / f"{sample}_read_counts_pcr_fragmented.tsv"
    if not fa.exists():
        raise FileNotFoundError(f"Missing shared templates for {sample}: {fa}\n"
                                 "Run template build in simulate_mouse_merged_insilicoseq.ipynb first.")
    if not tsv.exists():
        raise FileNotFoundError(f"Missing PCR-like read_counts for {sample}: {tsv}\n"
                                 "Run step 4 (build_pcr_fragmented_read_counts) first.")
print("templates + PCR-like fragmented read_counts OK for all samples")


templates + PCR-like fragmented read_counts OK for all samples


In [11]:
def run_iss_generate(sample, model=str(CUSTOM_MODEL_NPZ), sequence_type=SEQUENCE_TYPE, nproc=NPROC,
                      seed=SEED, compress=COMPRESS, force=FORCE,
                      fragment_length_mean=FRAGMENT_LENGTH_MEAN, fragment_length_sd=FRAGMENT_LENGTH_SD):
    templates_fa = SHARED_TEMPLATES_DIR / f"{sample}_templates.fasta"
    counts_tsv = COUNTS_DIR / f"{sample}_read_counts_pcr_fragmented.tsv"
    out_prefix = FASTQ_DIR / sample
    ext = ".fastq.gz" if compress else ".fastq"
    r1_out = Path(str(out_prefix) + f"_R1{ext}")
    r2_out = Path(str(out_prefix) + f"_R2{ext}")

    if r1_out.exists() and r2_out.exists() and not force:
        print(f"[{sample}] [skip] {r1_out.name} exists")
        return {"sample": sample, "status": "skipped"}

    stdout_path = LOGS_DIR / f"{sample}_iss.stdout.txt"
    stderr_path = LOGS_DIR / f"{sample}_iss.stderr.txt"

    cmd = [
        "iss", "generate",
        "--genomes", str(templates_fa),
        "--readcount_file", str(counts_tsv),
        "--sequence_type", sequence_type,
        "--model", model,
        "--cpus", str(nproc),
        "--output", str(out_prefix),
    ]
    if sequence_type == "metagenomics":
        cmd += ["--fragment-length", str(fragment_length_mean), "--fragment-length-sd", str(fragment_length_sd)]
    if seed is not None:
        cmd += ["--seed", str(seed)]
    if compress:
        cmd.append("--compress")

    print(f"[{sample}] [run] {' '.join(cmd)}")
    t0 = time.time()
    with open(stdout_path, "w") as out_h, open(stderr_path, "w") as err_h:
        proc = subprocess.Popen(cmd, stdout=out_h, stderr=err_h, text=True)
        print(f"  pid={proc.pid} stdout={stdout_path.name} stderr={stderr_path.name}")
        while True:
            rc = proc.poll()
            if rc is not None:
                break
            print(f"  still running: pid={proc.pid} elapsed={(time.time()-t0)/60:.1f} min", flush=True)
            time.sleep(30)
    elapsed = time.time() - t0
    if rc != 0:
        raise RuntimeError(f"iss generate failed for {sample} (exit {rc}); see {stderr_path}")

    n_r1 = sum(1 for _ in iter_fastq(r1_out))
    print(f"[{sample}] done: R1={n_r1:,} reads elapsed={elapsed/60:.1f} min")
    return {
        "sample": sample, "status": "done", "elapsed_sec": f"{elapsed:.1f}",
        "reads_generated": n_r1, "r1_fastq": str(r1_out), "r2_fastq": str(r2_out),
    }


In [ ]:
sim_rows = [run_iss_generate(sample) for sample in SAMPLES]

qc_path = QC_DIR / "simulate_qc.tsv"
done_rows = [r for r in sim_rows if r.get("status") == "done"]
if done_rows:
    with open(qc_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(done_rows[0].keys()), delimiter="\t")
        writer.writeheader()
        writer.writerows(done_rows)
    print(f"wrote {qc_path}")

total_generated = sum(r.get("reads_generated", 0) for r in sim_rows if r.get("status") == "done")
print(f"total simulated read pairs: {total_generated:,}")


[ERR346596] [run] iss generate --genomes /data/user/epishkin/results/ERP003950/simulated/insilicoseq/templates/ERR346596_templates.fasta --readcount_file /data/user/epishkin/results/ERP003950/simulated/insilicoseq_150bp_pcr_fragmented/read_counts/ERR346596_read_counts_pcr_fragmented.tsv --sequence_type metagenomics --model /data/user/epishkin/results/ERP003950/simulated/insilicoseq_150bp/model/ERP003950_mouse_miseq150.npz --cpus 8 --output /data/user/epishkin/results/ERP003950/simulated/insilicoseq_150bp_pcr_fragmented/fastq/ERR346596 --fragment-length 200 --fragment-length-sd 40 --seed 42 --compress
  pid=60694 stdout=ERR346596_iss.stdout.txt stderr=ERR346596_iss.stderr.txt
  still running: pid=60694 elapsed=0.0 min
  still running: pid=60694 elapsed=0.5 min
  still running: pid=60694 elapsed=1.0 min
  still running: pid=60694 elapsed=1.5 min
  still running: pid=60694 elapsed=2.0 min
  still running: pid=60694 elapsed=2.5 min
  still running: pid=60694 elapsed=3.0 min
  still running

### Notes

- Порядок модели: исходные real merged templates → PCR-like abundance overdispersion →
  fragmentation/metagenomics placement → 2x150 sequencing errors.
- Shared templates and original fixed-end counts remain in
  `results/ERP003950/simulated/insilicoseq/templates/`.
- New PCR-like counts are in
  `results/ERP003950/simulated/insilicoseq_150bp_pcr_fragmented/read_counts/`.
- Fastq output is in
  `results/ERP003950/simulated/insilicoseq_150bp_pcr_fragmented/fastq/{sample}_R1/_R2.fastq.gz`.
- Existing `insilicoseq_150bp/` results are not overwritten. If the old 150bp `.npz` model exists,
  this notebook reuses it; otherwise it trains a model inside its own `model/` directory.
